# CropClassifier Quickstart Guide

This notebook demonstrates how to use the CropClassifier package to automate crop mapping workflows. You can run the entire pipeline end-to-end with a single command or execute individual steps for fine-grained control over intermediate data.

 ## 1. Setup and Initialization

First, import the required classes and configure the core parameters such as your input shapefile, Google Earth Engine (GEE) project ID, target year, and output directory.

In [3]:
from pathlib import Path
from crop_classifier import CropClassifier, ModelType, OutputFormat

# Define input parameters
SHAPEFILE_PATH = "../data/raw/fields.shp"
GEE_PROJECT = "your-gee-project-id"
YEAR = 2024
OUTPUT_DIR = "../data"

# Initialize the classifier instance
classifier = CropClassifier(
    shapefile_path=SHAPEFILE_PATH,
    gee_project=GEE_PROJECT,
    year=YEAR,
    output_dir=OUTPUT_DIR,
    use_zonal_spectral=True,
    model_type=ModelType.FINETUNED,
    threshold=0.5,
    output_format=OutputFormat.TABLE,
    epsg_code="EPSG:32637"
)

## 2. Method 1: Running the Full Pipeline End-to-End

If you want to run the complete sequence (data downloading $\rightarrow$ feature extraction $\rightarrow$ model inference $\rightarrow$ result export) without manual intervention, call .run().

In [ ]:
# Execute the full pipeline
classifier.run()

The generated outputs will be stored automatically in structured subdirectories inside ./data:

 - data/raw/: Raw spectral and meteorological data downloaded from GEE.

 - data/processed/: Engineered feature dataset stored in Parquet format.

 - data/final/: Final prediction maps or tabular results.

## 3. Method 2: Running Pipeline Steps Individually

Running steps independently is useful when you want to inspect intermediate files, tweak feature engineering, or re-run predictions without redownloading satellite imagery.

### Step 1: Download Raw Data from GEE

Download multi-temporal satellite spectra and meteorological metrics for your input geometries.

In [ ]:
# Download spectral and meteo data to data/raw/
classifier.download_data()

### Step 2: Feature Engineering

Process the raw spectral and meteorological data into aggregated predictor features.

In [ ]:
# Process raw data into features (saved to data/processed/fields_input.parquet)
processed_features_path = classifier.build_features()

print(f"Features saved to: {processed_features_path}")

### Step 3: Model Inference & Result Export

Run model predictions on the engineered features and export the results to the target format (e.g., GeoTIFF or CSV/Table).

In [ ]:
# Generate predictions using default inputs from Step 2
classifier.predict()

### 4. Advanced: Overriding Paths for Custom Workflows

You can pass custom file paths to .build_features() and .predict(). This enables you to reuse existing feature files or point to external datasets outside the standard output directory structure.

In [ ]:
# Custom Feature Engineering using external raw inputs
custom_features = classifier.build_features(
    spectral_path="custom_data/spectral_2024.csv",
    meteo_path="custom_data/meteo_2024.csv",
    output_path="custom_data/processed_features.parquet"
)

# Custom Inference using non-standard feature input and output locations
classifier.predict(
    processed_path="custom_data/processed_features.parquet",
    output_prefix="custom_data/final_predictions/crop_map_2024"
)

## 5. Additional: Pixel-based determination of crop types

In some cases, you need to determine the type of crop without knowing where the fields are located. In crop_classifier The 'use_zonal_spectral' flag is responsible for this.

Note: this method requires more resources, as calculations are performed in each pixel of the image. Make sure that the shape-file's territory meets the limits of the Google Earth Engine.

In [ ]:
classifier = CropClassifier(
    shapefile_path=SHAPEFILE_PATH,
    gee_project=GEE_PROJECT,
    year=YEAR,
    output_dir=OUTPUT_DIR,
    use_zonal_spectral=False, # Turn the value to 'False' if you want to make pixel-based detection
    model_type=ModelType.FINETUNED,
    threshold=0.5,
    output_format=OutputFormat.TABLE,
    epsg_code="EPSG:32637"
)

# The rest of manipulations are similar
classifier.run()